In [1]:
1+1

2

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma



In [4]:
from langchain_community.llms import Ollama

# Connect to your local Llama 3.2 model
llm = Ollama(model="llama3.2")

response = llm.invoke("Explain what LangSmith does in simple terms.")
print(response)

C:\Users\Admin\AppData\Local\Temp\ipykernel_25076\3044438599.py:4: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.2")


LangSmith is a social media platform that allows users to create and share digital "cards" that showcase their interests, hobbies, and accomplishments. These cards can be used to connect with others who share similar passions and interests.

Think of LangSmith like a digital scrapbook or a digital business card, but instead of just showcasing your contact information, you can share all sorts of things that make you who you are. It's a way to showcase your personality, interests, and achievements in a fun and interactive way.

For example, someone might create a LangSmith card that showcases their love of music, their favorite books, or their passion for photography. They can then share this card with others on the platform, and connect with people who share similar interests.

LangSmith is often compared to other social media platforms like LinkedIn or Instagram, but it's more focused on showcasing personal interests and accomplishments rather than professional credentials.


In [5]:
loader = TextLoader("Knowledge.txt")

In [6]:
document = loader.load()

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
docs = splitter.split_documents(document)

In [8]:
docs

[Document(metadata={'source': 'Knowledge.txt'}, page_content='Johnathan "John" Miles is a man defined by a meticulous and aspirational "love" for James Bond, a passion that borders on lifestyle emulation. By day, John is a 44-year-old risk analyst in Chicago, a profession steeped in data and predictability, which he actively contrasts with the sophisticated, crisis-driven world of 007. This fixation is not just casual fandom; it is a dedicated curation of his personal aesthetic and environment. He invests significantly in the specific trappings of the character, favoring navy blue, single-breasted suits, brown suede chukka boots, and a specific Omega Seamaster Diver 300M, which he acquired after his last bonus. His evenings are often spent perfecting the Vesper martiniâ€”having researched the exact non-mainstream ingredients like Lillet Blancâ€”or streaming the John Barry instrumental scores. For John, the "love" for Bond is an appreciation for an ideal of precision, classic masculinit

In [9]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1025.87it/s]


In [10]:
vectorstore = Chroma.from_documents(docs,embedding=embedding,persist_directory="./chroma_db")

In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [12]:
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant. Use the following content to answer the question.
                                        Context: {context}
                                        Question:{question}
                                          give a concise, factual answer.

""")

In [13]:
rag_chain=(
    {
        "context":retriever | (lambda docs:"\n\n".join([d.page_content for d in docs])),
        "question":RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()

)

In [14]:
#testing 1 and output of the model

In [15]:
query = rag_chain.invoke("who is jhon")    

In [16]:
query                                      

'Johnathan "John" Miles is a 44-year-old risk analyst in Chicago.'

In [17]:
#testing 2 and output of the model

In [18]:
query = rag_chain.invoke("Jhon is the fan of which fictional character")


In [19]:
query

'James Bond.'